In [6]:
import re
import sys
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

import config
from gene_selectors import GeneSelector
from pipeline import data_prep, enrichment, plots
from pipeline.enrichment import gsea_run_dir, run_gsea_prerank
from pipeline.signatures import load_gsea
from viz_style import apply_style
apply_style()

In [2]:
dd = data_prep.load_disease_filtered() # Filter disease phenotype with OOD covariate distribution & too small samples sizes.
print(f'Final: {dd.Z_dis.shape[0]} samples, {len(np.unique(dd.dis_pheno))} phenotypes')

Final: 854 samples, 20 phenotypes


In [3]:
print(pd.Series(dd.dis_pheno).value_counts().to_string())

CAD_HF+                          112
Tuberculosis                     101
CAD_HF-                          100
ME/CFS                            90
Pancreatitis                      79
Pancreatic Cancer (Moore)         72
Pre-eclampsia                     59
Colorectal Cancer                 37
Lung Cancer                       30
Liver Cancer (Roskams-Hieter)     28
Esophagus Cancer (Chen)           25
Stomach Cancer                    24
MM                                17
Other Cancer                      16
HIV                               13
ICI-m                             11
ICI-treated Cancer                11
HIV + Tuberculosis                11
Liver Cancer (Chen)               10
MGUS                               8


## Ubiquity filter: target gene identification

Compute cross-disease ubiquity scores and identify which genes each condition zeros out in the GSEA ranking. Compare against known artifact-candidate families (ribo, OXPHOS, histone).

In [9]:
FILTER_CONDITIONS = [
    {'ubiquity_thr': None,  'ubiquity_abs_z': 0.5, 'label': 'no_filter'},
    {'ubiquity_thr': 0.5,   'ubiquity_abs_z': 0.5, 'label': 'ubiq50_abs5'},
    {'ubiquity_thr': 0.6,   'ubiquity_abs_z': 0.5, 'label': 'ubiq60_abs5'},
    {'ubiquity_thr': 0.7,   'ubiquity_abs_z': 0.5, 'label': 'ubiq70_abs5'},
]

for cond in FILTER_CONDITIONS:
    cond['gsea_dir'] = gsea_run_dir(cond['ubiquity_thr'], cond['ubiquity_abs_z'])
    print(f"{cond['label']:15s} → {cond['gsea_dir'].relative_to(config.ROOT)}")

no_filter       → Modeling/GSEA/no_filter
ubiq50_abs5     → Modeling/GSEA/ubiq50_abs5
ubiq60_abs5     → Modeling/GSEA/ubiq60_abs5
ubiq70_abs5     → Modeling/GSEA/ubiq70_abs5


In [ ]:
HISTONE_PAT = re.compile(r'^HIST[0-9]|^H[0-9][A-Z][A-Z0-9]|^H[0-9]-', re.I)
OXPHOS_PAT  = re.compile(r'^NDUF|^COX[0-9]|^ATP5|^UQCR|^SDHA|^SDHB|^SDHC|^SDHD')
RIBO_PAT    = re.compile(r'^RPS[0-9]|^RPL[0-9]|^RPLP')

def classify(s):
    if OXPHOS_PAT.match(s):  return 'oxphos'
    if RIBO_PAT.match(s):    return 'ribo'
    if HISTONE_PAT.match(s): return 'histone'
    return 'other'

gs = GeneSelector(dd.Z_dis, dd.dis_pheno, dd.gene_syms)
gene_cat = np.array([classify(s) for s in dd.gene_syms])

ubiq_rows = []
for cond in FILTER_CONDITIONS:
    thr = cond['ubiquity_thr']
    abs_z = cond['ubiquity_abs_z']
    if thr is None:
        ubiq_rows.append({'label': cond['label'], 'n_zeroed': 0,
                          'oxphos': 0, 'ribo': 0, 'histone': 0, 'other': 0})
        continue
    ubiq = gs.compute_ubiquity(abs_z_thr=abs_z)
    mask = ubiq >= thr
    ubiq_rows.append({
        'label':   cond['label'],
        'n_zeroed': int(mask.sum()),
        'oxphos':  int((mask & (gene_cat == 'oxphos')).sum()),
        'ribo':    int((mask & (gene_cat == 'ribo')).sum()),
        'histone': int((mask & (gene_cat == 'histone')).sum()),
        'other':   int((mask & (gene_cat == 'other')).sum()),
    })

ubiq_df = pd.DataFrame(ubiq_rows).set_index('label')
print('=== Genes zeroed out per filter condition ===')
print(ubiq_df.to_string())

n_oxphos = (gene_cat == 'oxphos').sum()
n_ribo   = (gene_cat == 'ribo').sum()
n_hist   = (gene_cat == 'histone').sum()
print(f'\nTotal genes in model: {len(dd.gene_syms)}')
print(f'Artifact-candidate families in model: oxphos={n_oxphos}, ribo={n_ribo}, histone={n_hist}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(FILTER_CONDITIONS) - 1, figsize=(10, 4), sharey=False)
cats = ['oxphos', 'ribo', 'histone', 'other']
colors = ['#16a085', '#c0392b', '#e67e22', '#95a5a6']

for ax, cond in zip(axes, [c for c in FILTER_CONDITIONS if c['ubiquity_thr'] is not None]):
    row = ubiq_df.loc[cond['label']]
    totals = [row[c] for c in cats]
    totals_pct = [row[c] / max(
        (gene_cat == c).sum(), 1) * 100 for c in cats]
    bars = ax.bar(cats, totals_pct, color=colors, alpha=0.85, edgecolor='none')
    for bar, n in zip(bars, totals):
        if n > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    str(n), ha='center', va='bottom', fontsize=10)
    ax.set_title(f"{cond['label']}\n({row['n_zeroed']} total zeroed)", fontsize=11)
    ax.set_ylabel('% of family zeroed out')
    ax.set_ylim(0, 105)

plt.suptitle('Fraction of each gene family zeroed in GSEA ranking', fontsize=13)
plt.tight_layout()
plt.show()

## GSEA run — all filter conditions

Run GSEA prerank for each condition and save to the corresponding directory. Skip if results already exist.

In [10]:
all_results = {}
unique_phenos = np.unique(dd.dis_pheno)

for cond in FILTER_CONDITIONS:
    label = cond['label']
    gsea_dir = cond['gsea_dir']
    thr = cond['ubiquity_thr']
    abs_z = cond['ubiquity_abs_z']

    existing = list(gsea_dir.glob('gsea_result_*.csv'))
    if existing:
        print(f"[{label}] loading {len(existing)} files from {gsea_dir.relative_to(config.ROOT)}")
        loaded = {}
        for ph in unique_phenos:
            df = load_gsea(ph, gsea_dir=gsea_dir)
            if df is not None:
                loaded[ph] = df
        all_results[label] = loaded
        print(f"  loaded: {len(loaded)}/{len(unique_phenos)} phenotypes")
        continue

    if thr is None:
        print(f"[{label}] no existing results found — run with ubiquity_thr=None first to generate baseline")
        all_results[label] = {}
        continue

    print(f"\n{'='*60}")
    print(f"[{label}] Running GSEA → {gsea_dir.relative_to(config.ROOT)}")
    print(f"{'='*60}")
    results = run_gsea_prerank(
        dd.Z_dis, dd.dis_pheno, dd.gene_syms,
        outdir=gsea_dir,
        ubiquity_thr=thr,
        ubiquity_abs_z=abs_z,
    )
    all_results[label] = results
    print(f"[{label}] Done — {len(results)} phenotypes written")

[no_filter] loading 22 files from Modeling/GSEA/no_filter
  loaded: 20/20 phenotypes
[ubiq50_abs5] loading 20 files from Modeling/GSEA/ubiq50_abs5
  loaded: 20/20 phenotypes
[ubiq60_abs5] loading 20 files from Modeling/GSEA/ubiq60_abs5
  loaded: 20/20 phenotypes
[ubiq70_abs5] loading 20 files from Modeling/GSEA/ubiq70_abs5
  loaded: 20/20 phenotypes


## GSEA run — with_rare (optional, side-by-side)

In [13]:
from pipeline import scoring

WITH_RARE_DIR = config.GSEA_DIR / 'with_rare'
existing = list(WITH_RARE_DIR.glob('gsea_result_*.csv'))
unique_phenos = np.unique(dd.dis_pheno)

if existing:
    print(f"[with_rare] loading {len(existing)} files from {WITH_RARE_DIR.relative_to(config.ROOT)}")
    results_with_rare = {}
    for ph in unique_phenos:
        df = load_gsea(ph, gsea_dir=WITH_RARE_DIR)
        if df is not None:
            results_with_rare[ph] = df
    print(f"  loaded: {len(results_with_rare)}/{len(unique_phenos)} phenotypes")
else:
    Z_with_rare = scoring.score_disease_with_rare(dd)
    print(f"\n{'='*60}")
    print(f"[with_rare] Running GSEA → {WITH_RARE_DIR.relative_to(config.ROOT)}")
    print(f"{'='*60}")
    results_with_rare = run_gsea_prerank(
        Z_with_rare, dd.dis_pheno, dd.gene_syms,
        outdir=WITH_RARE_DIR,
        ubiquity_thr=None,
    )
    print(f"[with_rare] Done — {len(results_with_rare)} phenotypes written")

all_results['with_rare'] = results_with_rare

Engine loaded from /project/cfRNA_NormativeModeling/Modeling/../Modeling/engine_state/  (19538 fitted genes)
Loaded reference: 559 rare genes
Z matrix: (854, 20097)  (9.5s)

[with_rare] Running GSEA → Modeling/GSEA/with_rare
CAD_HF+                  :  937 sig  (NES>0:  22  NES<0: 915)
CAD_HF-                  :  884 sig  (NES>0:   5  NES<0: 879)
Colorectal Cancer        :  382 sig  (NES>0:  23  NES<0: 359)
Esophagus Cancer (Chen)  :  710 sig  (NES>0: 150  NES<0: 560)
HIV                      :  425 sig  (NES>0: 377  NES<0:  48)
HIV + Tuberculosis       :  417 sig  (NES>0: 332  NES<0:  85)
ICI-m                    :  506 sig  (NES>0: 331  NES<0: 175)
ICI-treated Cancer       : 1329 sig  (NES>0: 223  NES<0: 1106)
Liver Cancer (Chen)      :  700 sig  (NES>0: 230  NES<0: 470)
Liver Cancer (Roskams-Hieter):  264 sig  (NES>0: 246  NES<0:  18)
Lung Cancer              :  503 sig  (NES>0:  42  NES<0: 461)
ME/CFS                   :  240 sig  (NES>0:  14  NES<0: 226)
MGUS                     :

In [ ]:
sample_sizes = {ph: int((dd.dis_pheno == ph).sum()) for ph in unique_phenos}

for cond in FILTER_CONDITIONS:
    label = cond['label']
    gsea_dir = cond['gsea_dir']
    fig_dir = gsea_dir / 'Figures' / 'gsea_dotplot'

    res = all_results.get(label, {})
    if not res:
        print(f"[{label}] no results — skipping plots")
        continue

    print(f"[{label}] saving dotplots → {fig_dir.relative_to(config.ROOT)}")
    plots.plot_gsea_dotplots(res, fig_dir=fig_dir, sample_sizes=sample_sizes)

[no_filter] saving dotplots → Modeling/GSEA/no_filter/Figures/gsea_dotplot
[CAD_HF+] GSEA Plots generated successfully.
[CAD_HF-] GSEA Plots generated successfully.
[Colorectal Cancer] GSEA Plots generated successfully.
[Esophagus Cancer (Chen)] GSEA Plots generated successfully.
[HIV] GSEA Plots generated successfully.
[HIV + Tuberculosis] GSEA Plots generated successfully.
[ICI-m] GSEA Plots generated successfully.
[ICI-treated Cancer] GSEA Plots generated successfully.
[Liver Cancer (Chen)] GSEA Plots generated successfully.
[Liver Cancer (Roskams-Hieter)] GSEA Plots generated successfully.
[Lung Cancer] GSEA Plots generated successfully.
[ME/CFS] GSEA Plots generated successfully.
[MGUS] GSEA Plots generated successfully.
[MM] GSEA Plots generated successfully.
[Other Cancer] GSEA Plots generated successfully.
[Pancreatic Cancer (Moore)] GSEA Plots generated successfully.
[Pancreatitis] GSEA Plots generated successfully.
[Pre-eclampsia] GSEA Plots generated successfully.
[Stomach C

In [ ]:
sample_sizes = {ph: int((dd.dis_pheno == ph).sum()) for ph in unique_phenos}

for pheno in unique_phenos : 
    res = all_results['with_rare'][pheno]
    if not res:
        print(f"[{label}] no results — skipping plots")
        continue
    fig_dir = WITH_RARE_DIR / 'Figures' / 'gsea_dotplot'
    plots.plot_gsea_dotplots(results, fig_dir=fig_dir, sample_sizes=sample_sizes)

## Cross-condition comparison

In [ ]:
import re as _re

ARTIFACT_PAT = _re.compile(
    r'histone|chromatin|nucleosome'
    r'|mitochond|oxidative phosph|electron transport|respiratory chain'
    r'|ribosom|rRNA|translation elongation|translation initiation',
    _re.IGNORECASE,
)


def summarise_results(results):
    """Summarise GSEA results dict {phenotype: DataFrame}.

    Returns DataFrame indexed by phenotype with columns:
      n_up, n_dn, n_total         — significant term counts by NES direction
      n_artifact, artifact_frac   — artifact-candidate terms (count and fraction of n_total)
    """
    rows = []
    for ph, df in results.items():
        df = df.copy()
        df['NES'] = pd.to_numeric(df['NES'], errors='coerce')
        n_up  = int((df['NES'] > 0).sum())
        n_dn  = int((df['NES'] < 0).sum())
        n_tot = n_up + n_dn
        n_art = int(df['Term'].str.contains(ARTIFACT_PAT, na=False).sum())
        rows.append({
            'phenotype':    ph,
            'n_up':         n_up,
            'n_dn':         n_dn,
            'n_total':      n_tot,
            'n_artifact':   n_art,
            'artifact_frac': round(n_art / n_tot, 3) if n_tot > 0 else 0.0,
        })
    return pd.DataFrame(rows).set_index('phenotype').sort_index()


summaries = {label: summarise_results(res) for label, res in all_results.items()}

print('=== Total significant terms per condition ===')
totals = pd.DataFrame({lbl: s['n_total'] for lbl, s in summaries.items()})
totals.loc['TOTAL'] = totals.sum()
print(totals.to_string())

print()
print('=== Artifact terms: count per condition ===')
arts_n = pd.DataFrame({lbl: s['n_artifact'] for lbl, s in summaries.items()})
arts_n.loc['TOTAL'] = arts_n.sum()
print(arts_n.to_string())

print()
print('=== Artifact terms: fraction of significant terms ===')
arts_f = pd.DataFrame({lbl: s['artifact_frac'] for lbl, s in summaries.items()})
arts_f.loc['MEAN'] = arts_f.mean()
print(arts_f.round(3).to_string())

print()
print('=== Change vs no_filter (delta n_total = condition - no_filter) ===')
ref = summaries.get('no_filter')
if ref is not None:
    non_ref = [lbl for lbl in summaries if lbl != 'no_filter']
    if non_ref:
        delta = pd.DataFrame({
            lbl: summaries[lbl]['n_total'] - ref['n_total']
            for lbl in non_ref
        })
        delta.loc['TOTAL'] = delta.sum()
        print(delta.to_string())


In [ ]:
import glob as _glob

DESEQ2_DIR = config.MODELING_DIR / 'Benchmark' / 'deseq2_results'
_deseq_gsea = {}
for _f in sorted(_glob.glob(str(DESEQ2_DIR / 'gsea_result_*.csv'))):
    _ph = Path(_f).stem.replace('gsea_result_', '').replace('_', ' ')
    _deseq_gsea[_ph] = pd.read_csv(_f)

if _deseq_gsea:
    all_results['deseq2'] = _deseq_gsea
    print(f"Added 'deseq2' to all_results: {len(_deseq_gsea)} phenotypes")
else:
    print("No DESeq2 GSEA files found — run DESeq2 GSEA cells first")

print("Keys in all_results:", list(all_results.keys()))

summaries = {label: summarise_results(res) for label, res in all_results.items()}

print()
print('=== Total significant terms (including DESeq2) ===')
totals = pd.DataFrame({lbl: s['n_total'] for lbl, s in summaries.items()})
totals.loc['TOTAL'] = totals.sum()
print(totals.to_string())

print()
print('=== Artifact terms: count ===')
arts_n = pd.DataFrame({lbl: s['n_artifact'] for lbl, s in summaries.items()})
arts_n.loc['TOTAL'] = arts_n.sum()
print(arts_n.to_string())

print()
print('=== Artifact terms: fraction of significant terms ===')
arts_f = pd.DataFrame({lbl: s['artifact_frac'] for lbl, s in summaries.items()})
arts_f.loc['MEAN'] = arts_f.mean()
print(arts_f.round(3).to_string())


In [ ]:
import matplotlib.pyplot as plt

# Grouped bar: total sig terms per condition including deseq2
labels = list(summaries.keys())
phenos_u = sorted(set.union(*[set(s.index) for s in summaries.values()]))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, title in [
    (axes[0], 'n_total',    'Total significant GSEA terms'),
    (axes[1], 'n_artifact', 'Artifact-candidate terms (ribo/OXPHOS)'),
]:
    x = np.arange(len(phenos_u))
    width = 0.8 / len(labels)
    for i, label in enumerate(labels):
        vals = [summaries[label].get(ph, {}).get(col, 0)
                if ph in summaries[label].index else 0
                for ph in phenos_u]
        ax.bar(x + i * width, vals, width, label=label, alpha=0.85)
    ax.set_xticks(x + width * (len(labels) - 1) / 2)
    ax.set_xticklabels([p[:18] for p in phenos_u], rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Normative (filter conditions) vs DESeq2 — GSEA term counts', fontsize=13)
plt.tight_layout()
plt.show()
